# Single-GPU vs. multi-GPU (DDP) CNN training on CIFAR-10

This notebook runs the scripts in this folder and compares wall-clock training time.
Set `DATA_DIR` and `CKPT_DIR` to locations on your cluster's scratch filesystem.

In [ ]:
DATA_DIR = "./data"
CKPT_DIR = "./checkpoints"
EPOCHS, SAVE_EVERY, BATCH_SIZE = 150, 50, 64

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Number of CUDA devices:", torch.cuda.device_count())

In [ ]:
!nvidia-smi

## 1. Single-GPU baseline (`cnn.py`)

In [ ]:
!time python3 cnn.py {EPOCHS} {SAVE_EVERY} --batch_size {BATCH_SIZE} --data-dir {DATA_DIR} --ckpt-path {CKPT_DIR}/cnn.pth

In [ ]:
!python3 evaluate.py {CKPT_DIR}/cnn.pth --data-dir {DATA_DIR}

## 2. Multi-GPU with DistributedDataParallel (`parallel-cnn.py`)

In [ ]:
!time python3 parallel-cnn.py {EPOCHS} {SAVE_EVERY} --batch_size {BATCH_SIZE} --data-dir {DATA_DIR} --ckpt-path {CKPT_DIR}/parallel-cnn.pt

In [ ]:
!python3 evaluate.py {CKPT_DIR}/parallel-cnn.pt --data-dir {DATA_DIR}

## Results (2 GPUs, 150 epochs, batch size 64)

| Run | real | user | sys |
|---|---|---|---|
| Single GPU (`cnn.py`) | 26m 15s | 25m 32s | 0m 54s |
| DDP, 2 GPUs (`parallel-cnn.py`) | 20m 07s | 190m 18s | 1m 12s |

- **Real time:** DDP finished faster because each GPU processes half of every epoch.
- **User time:** much higher for DDP because it runs one process per GPU (plus data loading), and each process adds to the total CPU time.
- **Sys time:** slightly higher for DDP, due to inter-process and GPU-to-GPU communication overhead.

Note: the single-GPU timing included the first-time CIFAR-10 download.

## 3. PyTorch Lightning DDP (`ddp_lightning.py`)

In [ ]:
!time python3 ddp_lightning.py --epochs 5 --devices 2 --data-dir {DATA_DIR} --ckpt-path {CKPT_DIR}/lightning-cnn.pt

## Sources
- https://github.com/pytorch/examples/blob/main/distributed/ddp-tutorial-series/multigpu.py
- https://www.youtube.com/watch?v=-LAtx9Q6DA8